<a id="top"></a>
# House Sales: preparazione dati e primo modello di regressione (esercitazione)

Nel modulo 1 hai usato il dataset House Sales per inquadrare il problema: stimare il **prezzo di vendita** delle case in una certa area.  
Nel modulo 2, con il caso Bank Marketing, hai lavorato sulla preparazione di un dataset per un problema di **classificazione binaria**.

Qui resti nel contesto naturale di House Sales: **regressione sul prezzo**.  
L’obiettivo è replicare in versione compatta la pipeline di preparazione dati vista nel marketing bancario e collegarla a un primo modello lineare.

<a id="indice"></a>
## Indice

- [Obiettivo dell'esercitazione](#obiettivo-esercitazione)
- [Caricamento del dataset House Sales](#caricamento-dataset)
- [Target di regressione e scelta delle feature](#target-feature)
- [Encoding delle variabili categoriali](#encoding-categoriche)
- [Imputazione e normalizzazione](#imputazione-scaling)
- [Costruzione del dataset modellabile](#dataset-finale)
- [Modello lineare: LinearRegression](#modello-lineare)
- [Spunti di riflessione](#spunti-riflessione)


<a id="obiettivo-esercitazione"></a>
## Obiettivo dell'esercitazione

Nel caso Bank Marketing ti sei concentrato su una **classificazione**: prevedere se un cliente sottoscrive o meno un deposito (output binario 0/1).  
Nel caso House Sales torni al problema di **regressione**: il modello prevede un **numero continuo** (il prezzo), non una classe.

Riassumendo in due righe la differenza:

- **Classificazione**: l’output è una categoria (es. _sì/no_, _classe A/B/C_).  
- **Regressione**: l’output è un valore continuo (es. prezzo in euro).

In questa esercitazione:

- prepari le feature in modo coerente con quanto fatto sul marketing (imputazione + normalizzazione robusta);
- alleni un modello lineare (`LinearRegression`) per prevedere il prezzo;
- ragioni brevemente su come questo si confronta con un modello ad alberi (`RandomForestRegressor`) rispetto al **classifier** che hai appena visto.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Nota importante**

Un modello come `RandomForestRegressor` lavora per natura su valori non scalati e gestisce bene le non linearità.  
La famiglia lineare (`LogisticRegression` per la classificazione, `LinearRegression` per la regressione) è invece più sensibile alla scala delle variabili: per questo qui ti chiediamo di **normalizzare** le feature numeriche prima di allenare il modello.

</div>


<a id="caricamento-dataset"></a>
## Caricamento del dataset House Sales

Userai lo stesso dataset del notebook House Sales del modulo 1.  
Il dataset è su OpenML (id 42092) e si può scaricare con `fetch_openml` di scikit-learn.

- `fetch_openml` scarica un dataset da OpenML e restituisce un oggetto da cui puoi ricavare un DataFrame.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html

Nel blocco seguente carichi i dati in un DataFrame `house` e dai un’occhiata alle prime righe.


In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  # House sales
house = raw.frame.copy()

print("Dimensioni del dataset (righe, colonne):", house.shape)
house.head()


<a id="target-feature"></a>
## Target di regressione e scelta delle feature

Il target naturale è il **prezzo di vendita** della casa. Nel dataset la colonna del prezzo si chiama normalmente `price`.

Qui ti serve:

1. identificare la colonna target (`price`);  
2. rimuoverla dal set di feature;  
3. scegliere come feature le colonne numeriche “sensate” per la regressione.

Per semplificare, puoi:

- usare tutte le colonne numeriche come candidate;  
- escludere solo la colonna `price` e gli identificativi puri come `id` o `date` se presenti.

Strumenti utili:

- `DataFrame.select_dtypes` per selezionare le colonne per tipo.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Nel codice qui sotto:

1. completa il nome della colonna del prezzo dove vedi `___`;  
2. usa `select_dtypes` per selezionare le colonne numeriche;  
3. verifica quante feature numeriche rimangono dopo aver escluso il target e gli identificativi.

</div>


In [ ]:
house_prep = house.copy()

# Colonna del prezzo (target di regressione)
target_col = "___"  # TODO: inserisci il nome corretto della colonna del prezzo (es. "price")

# Colonne numeriche candidate
numeric_cols = house_prep.select_dtypes(include="number").columns.tolist()

# Escludiamo target e identificativi se presenti
cols_to_drop = [target_col, "id", "date"]
cols_to_drop = [c for c in cols_to_drop if c in numeric_cols]

feature_cols = [c for c in numeric_cols if c not in cols_to_drop]

X = house_prep[feature_cols].copy()
y = house_prep[target_col].copy()

print("Target:", target_col)
print("Numero di feature numeriche:", len(feature_cols))
X.head()


<a id="encoding-categoriche"></a>
## Encoding delle variabili categoriali

Nel dataset House Sales ci sono poche variabili non numeriche. La più evidente è `zipcode`, che rappresenta la zona della casa.

Per restare vicini allo schema visto nel marketing bancario puoi trasformare `zipcode` in una serie di colonne 0/1 con un **one-hot encoding**:

- `pandas.get_dummies` crea una colonna per ogni categoria.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html

In questa esercitazione puoi usare una scelta pragmatica:

- se `zipcode` esiste, applica `get_dummies` solo a quella colonna;  
- lascia invariato il resto.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Nel codice seguente:

1. controlla se `zipcode` è presente nel DataFrame `house_prep`;  
2. applica `get_dummies` solo se la colonna esiste;  
3. osserva come cambia la forma del DataFrame delle feature (`X_enc`).

</div>


In [ ]:
X_enc = X.copy()

# Colonne nominali da espandere (se presenti)
nominal_cols = [col for col in ["zipcode"] if col in house_prep.columns]

print("Colonne nominali da espandere:", nominal_cols)

if nominal_cols:
    X_enc = pd.get_dummies(
        X_enc,
        columns=nominal_cols,
        drop_first=True  # rimuove una categoria di riferimento per ogni variabile
    )

print("Forma delle feature dopo l'encoding:", X_enc.shape)
X_enc.head()


In [ ]:
# Controllo dei valori mancanti dopo l'encoding
X_enc.isna().mean().sort_values(ascending=False).head(10)


<a id="imputazione-scaling"></a>
## Imputazione e normalizzazione

Come nel modulo 2 sul marketing bancario, separi due passaggi:

1. **Imputazione** dei valori mancanti con `IterativeImputer`.  
2. **Normalizzazione robusta** delle feature non binarie con `RobustScaler`.

Richiamo veloce:

- `IterativeImputer` stima i valori mancanti usando modelli regressivi sulle altre colonne.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html  
- `RobustScaler` scala le feature usando mediana e IQR, riducendo l’effetto degli outlier.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Osserva il codice di imputazione e normalizzazione qui sotto e collega ogni passaggio a uno dei concetti che hai visto nel caso Bank Marketing:

- dove impariamo i parametri di imputazione e scaling?  
- dove applichiamo le trasformazioni ai dati?  
- perché lasciamo in pace le colonne puramente binarie (0/1)?

</div>


In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

# Imputazione dei valori mancanti
imputer = IterativeImputer(random_state=42)

X_imputed_array = imputer.fit_transform(X_enc)

# Ricostruiamo un DataFrame per mantenere nomi di colonne e indice
X_imputed = pd.DataFrame(
    X_imputed_array,
    columns=X_enc.columns,
    index=X_enc.index
)

# Controllo rapido: nessun valore mancante
X_imputed.isna().sum().sum()


In [ ]:
from sklearn.preprocessing import RobustScaler
import numpy as np

# Identificazione delle colonne binarie (solo valori 0/1)
binary_cols = []
for col in X_imputed.columns:
    unique_vals = pd.unique(X_imputed[col])
    if set(unique_vals).issubset({0, 1, 0.0, 1.0}):
        binary_cols.append(col)

numeric_cols = [col for col in X_imputed.columns if col not in binary_cols]

print("Numero di feature binarie lasciate inalterate:", len(binary_cols))
print("Numero di feature scalate:", len(numeric_cols))

scaler = RobustScaler()

X_scaled = X_imputed.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_imputed[numeric_cols])

X_scaled.head()


<a id="dataset-finale"></a>
## Costruzione del dataset modellabile

A questo punto hai:

- `X_imputed`: feature numeriche senza valori mancanti, non scalate;
- `X_scaled`: le stesse feature, con scaling robusto sulle colonne non binarie;
- `y`: la variabile target continua `price` (o nome equivalente).

Per il modello di regressione userai `X_scaled` come matrice di feature e `y` come target.  
In questa fase **non** fai ancora train/test split: la parte di validazione verrà gestita in un modulo dedicato.


In [ ]:
df_model_scaled = X_scaled.copy()
df_model_scaled[target_col] = y

print("Forma df_model_scaled:", df_model_scaled.shape)
df_model_scaled.head()


<a id="modello-lineare"></a>
## Modello lineare: LinearRegression

Ora puoi allenare un primo modello di regressione usando `LinearRegression` di scikit-learn:

- usa tutte le righe di `df_model_scaled` per l’addestramento, senza suddividere in train/test;
- sfrutta le feature scalate (`X_scaled`);  
- addestra il modello e osserva alcune previsioni di prezzo a confronto con i valori reali.

Riferimenti:

- `sklearn.linear_model.LinearRegression`  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

Per fare da ponte con quanto hai già visto:

- nel caso Bank Marketing: `RandomForestClassifier` → classificazione (output binario o multi-classe);  
- qui potresti usare `RandomForestRegressor` per la regressione (output continuo), ma in questa esercitazione ti concentri su `LinearRegression` per collegarti al tema della **normalizzazione**.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Nel blocco seguente:

1. completa le righe che trasformano `df_model_scaled` in `X_final` (feature) e `y_final` (target);  
2. allena una `LinearRegression` sui dati completi;  
3. guarda le prime 10 previsioni e confrontale con i prezzi reali.

Domande guida:

- l’output del modello è **continuo** o binario?  
- ha senso confrontare direttamente le previsioni con i prezzi reali in una tabella? cosa noti?

</div>


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Matrice delle feature e target (dataset scalato)
X_final = df_model_scaled.drop(columns=["___"])  # TODO: inserisci il nome della colonna target (es. "price")
y_final = df_model_scaled["___"]                # TODO: stesso nome della colonna target

# Modello di regressione lineare
lin_reg = LinearRegression()

lin_reg.fit(X_final, y_final)

# Previsioni sullo stesso dataset (la parte di validazione verrà dopo)
y_pred = lin_reg.predict(X_final)

preview = pd.DataFrame({
    "price_true": y_final.iloc[:10].values,
    "price_pred": np.round(y_pred[:10], 1)
})

preview


<a id="spunti-riflessione"></a>
## Spunti di riflessione

Qualche domanda per chiudere l’esercitazione:

- Rispetto al caso Bank Marketing, cosa cambia nel modo di leggere le previsioni?  
  (qui hai numeri continui, non probabilità o classi)  
- Guardando `price_true` e `price_pred`, i valori sono della stessa ordine di grandezza?  
- In quali situazioni ti aspetteresti che un modello come `RandomForestRegressor` possa fare meglio di una `LinearRegression`?  
  Pensa alla presenza di non linearità e interazioni complesse tra le feature.

Questa esercitazione ti serve per fissare l’idea che la pipeline di preparazione dati può essere quasi identica tra casi diversi.  
A cambiare è soprattutto **la natura del target** (classe vs valore continuo) e la scelta del modello al fondo della catena.
